# Coop Case — Q4.4: Where Does the Money Actually Come From?

**Question:** Which categories actually drive Coop's profit? A blunt Pareto cut on the product
portfolio -- different lens than Q2 (sustainability share) and Q3 (customer value): this is pure
profit concentration across the 134 product categories.


## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)
df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]

print(df.shape)
df.head()

## 2. Profit by category, ranked

In [ ]:
cat_df = df.dropna(subset=["ItemCategoryName"])
cat_profit = cat_df.groupby("ItemCategoryName", observed=True)["profit"].sum().sort_values(ascending=False)

total_profit = cat_profit.sum()
print(f"Total categories: {len(cat_profit)}")
print(f"Total profit (2 months, both stores): {total_profit:,.0f} SEK\n")
cat_profit.head(15)


## 3. The Pareto curve

How many categories does it take to reach 50% / 80% / 90% of total profit?


In [ ]:
cum_pct = cat_profit.cumsum() / total_profit * 100
n_categories = len(cat_profit)

for pct in [50, 80, 90]:
    n = (cum_pct >= pct).values.argmax() + 1
    print(f"{n} categories ({n/n_categories:.1%} of all {n_categories} categories) drive {pct}% of total profit")


In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5.5))
x = np.arange(1, len(cat_profit) + 1)
ax1.bar(x, cat_profit.values, color="#8FA097", width=1.0)
ax1.set_xlabel("Category rank (by profit)")
ax1.set_ylabel("Profit per category (SEK)", color="#8FA097")

ax2 = ax1.twinx()
ax2.plot(x, cum_pct.values, color="#22B573", linewidth=2.5)
ax2.axhline(80, color="#E8734A", linestyle="--", linewidth=1)
ax2.set_ylabel("Cumulative % of total profit", color="#22B573")
ax2.set_ylim(0, 105)

ax1.set_title("Category profit Pareto: a small slice of the catalog drives most of the profit")
plt.tight_layout()
plt.show()


## 4. Any categories that are actually losing money?

In [ ]:
loss_making = cat_profit[cat_profit < 0]
print(f"Loss-making categories: {loss_making.shape[0]}")
loss_making.round(0)


## 5. Takeaways

- **Profit is heavily concentrated in a small slice of the 134-category catalog**:
  - **15 categories (11.2%)** drive **50%** of total profit
  - **39 categories (29.1%)** drive **80%**
  - **53 categories (39.6%)** drive **90%**

  In other words, roughly **60% of Coop's product categories together contribute only ~10% of
  total profit** in this sample. This is a classic long-tail retail pattern, but worth stating
  explicitly with real numbers rather than assuming it.

- **The top 10 categories by profit** are dominated by fresh and everyday staples: **GRÖNSAKER**
  (vegetables, 606K SEK), **OST** (cheese, 548K), **MEJERI** (dairy, 461K), **MATCHARK** (deli meats,
  365K), **GODIS** (candy, 297K), **MJUKT MATBRÖD** (soft bread, 248K), **FRYST FÄRDIGLAGA** (frozen
  ready-made, 225K), **SNACKS** (209K), **FÄRDIGLAGAD MAT** (ready meals, 204K), **PÅLÄGGSCHARK**
  (sandwich meats, 199K). These aren't necessarily the highest-*margin* categories (see Q4.3 --
  meat and fruit are actually low-margin) -- they're on top here because of sheer volume, which is
  the point: profit concentration is a volume story as much as a margin story.

- **Only 4 categories are net loss-making**, and all are negligible in size (SPEL -24 SEK,
  OMBUDSTJÄNSTER -65 SEK, and two effectively-zero categories) -- not a real problem area,
  more a rounding/administrative artifact.

- **What this means for Coop**: the long tail of ~80 categories contributing under 20% of profit
  is worth a portfolio review -- not necessarily to cut them (some may serve a "one-stop-shop"
  completeness purpose that drives footfall even if individually low-profit), but it's worth knowing
  which specific categories are carrying the business before making assortment, shelf-space, or
  promotional-budget decisions. The top ~15 categories deserve disproportionate protection and
  attention (stockouts here hurt far more than in the long tail).
